### Model Selection / Cross-Validation

Used informally throughout this series ("held out on val", "stratify=df['typology']" in the fraud project) without ever explaining the mechanics as their own topic. Covers k-fold, stratified k-fold, nested CV.

## K-Fold Cross-Validation

#### 0. Core idea

A single train/val split gives one noisy estimate of generalization performance, it depends heavily on which rows happened to land in val. K-fold CV: split the data into k equal folds, for each fold, train on the other k-1 folds and validate on the held-out one, average the k validation scores. Every row gets used for validation exactly once, and for training k-1 times.

Toy setup, 10 examples, k=5 (2 examples per fold):
```
fold0=[0,1]  fold1=[2,3]  fold2=[4,5]  fold3=[6,7]  fold4=[8,9]

split 1: train on folds 1-4, validate on fold 0
split 2: train on folds 0,2,3,4, validate on fold 1
... (5 splits total, one per fold as the held-out validation set)
```
Final estimate: average of the 5 validation scores, plus the standard deviation across them, which itself is useful information, a high spread across folds means the model's performance is unstable/sensitive to which rows it happened to see, a low spread means the estimate is reliable.

In [ ]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(0)
X = rng.normal(size=(10, 3))
y = (X[:, 0] > 0).astype(int)

kf = KFold(n_splits=5)
for fold_num, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"fold {fold_num}: train_idx={train_idx}, val_idx={val_idx}")

scores = cross_val_score(LogisticRegression(), X, y, cv=5)
print("\nfold scores:", scores.round(3))
print("mean:", scores.mean().round(3), "| std:", scores.std().round(3))

## Stratified K-Fold

#### 0. The problem plain k-fold has with imbalanced classes

Worked example, 10 examples, labels [0,0,0,0,0,0,0,0,1,1] (imbalanced, only 2 of class 1, at indices 8 and 9). Plain k-fold, splitting sequentially into 5 folds of 2:
```
fold0=[0,1] fold1=[2,3] fold2=[4,5] fold3=[6,7] fold4=[8,9]
```
fold4 contains BOTH class-1 examples, and only them. Consequences:
- When fold4 is the validation set: the training data (folds 0-3) has ZERO class-1 examples. The model never sees class 1 during training for this split, cannot possibly predict it.
- When any of folds 0-3 is the validation set: that validation set has ZERO class-1 examples. Cannot evaluate class-1 recall/precision at all in those 4 of the 5 splits.

Stratified k-fold fixes this by ensuring each fold gets a proportional share of each class, the 2 class-1 examples get spread across different folds instead of landing together by chance, e.g. one in fold0, one in fold2, none of the folds end up with zero class-1 representation on both sides of the split. Directly relevant to the fraud-theme-detection project's own train_test_split call, which used `stratify=df['typology']` for exactly this reason, keeping the 200-per-theme balance intact across train/val/test.

In [ ]:
from sklearn.model_selection import StratifiedKFold

y_imbalanced = np.array([0,0,0,0,0,0,0,0,1,1])
X_imbalanced = np.arange(10).reshape(-1, 1)

print("plain KFold:")
for train_idx, val_idx in KFold(n_splits=5).split(X_imbalanced):
    val_labels = y_imbalanced[val_idx]
    print(f"  val_idx={val_idx}, val labels={val_labels}")

print("\nStratifiedKFold:")
for train_idx, val_idx in StratifiedKFold(n_splits=5).split(X_imbalanced, y_imbalanced):
    val_labels = y_imbalanced[val_idx]
    print(f"  val_idx={val_idx}, val labels={val_labels}")

## Nested Cross-Validation

#### 0. The problem plain CV has when you are ALSO tuning hyperparameters

If you use k-fold CV to pick the best hyperparameters (as in `hyperparameter-tuning.ipynb`'s GridSearchCV/RandomizedSearchCV), and then report THAT SAME cross-validated score as your estimate of how well the model generalizes, the estimate is optimistic. The hyperparameter search already saw and optimized against those exact validation folds, the reported score reflects some amount of fitting to the validation folds themselves, not genuinely unseen data, the same leakage principle as evaluating on data the model was tuned against, one level removed from training data itself but still not fully honest.

#### 1. The fix: two nested loops

Outer loop: k-fold split for the FINAL performance estimate. Inner loop: within each outer training fold, run a full hyperparameter search (its own k-fold CV) to pick the best config, then evaluate that config on the outer loop's held-out fold, which the inner search never touched.
```
for outer_train, outer_test in outer_kfold.split(X):
    best_params = grid_search(outer_train)     # inner CV, tunes hyperparameters
    score = evaluate(best_params, outer_test)  # outer_test was NEVER used by the inner search
    outer_scores.append(score)

final estimate = mean(outer_scores)
```
Expensive, k_outer * k_inner total model fits instead of just k, but it is the honest way to report generalization performance when hyperparameter tuning is part of the pipeline being evaluated, not an afterthought bolted onto a single CV run.

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score

param_grid = {"C": [0.1, 1.0, 10.0]}

# nested CV: GridSearchCV (inner loop) wrapped inside cross_val_score (outer loop)
inner_search = GridSearchCV(LogisticRegression(), param_grid, cv=3)
nested_scores = cross_val_score(inner_search, X, y, cv=3)

print("nested CV scores (honest generalization estimate):", nested_scores.round(3))
print("mean:", nested_scores.mean().round(3))